# Kathmandu Urban Growth Timelapse (1950–2025)

This project visualizes the spatial distribution and estimated historical expansion of buildings in Kathmandu, Nepal, from 1950 to 2025.

The workflow uses OpenStreetMap building footprints, OSMnx, GeoPandas, Matplotlib, and Contextily to generate a spatially animated representation of urban development.

## Objectives
- Retrieve building footprint data from OpenStreetMap.
- Project building geometries into UTM coordinates.
- Develop a distance-based model to assign estimated construction years.
- Generate time-series maps at two-year intervals.
- Export the visualization as GIF and MP4 animations.

## Important methodological note

The construction years are model-generated estimates based on distance from a selected city centre. They are not verified historical construction dates. Therefore, the animation represents a hypothetical spatial growth scenario, not measured historical urban expansion.

In [ ]:
# CELL 1: INSTALL

!pip install -q geopandas matplotlib osmnx imageio tqdm contextily

In [ ]:
# CELL 2: IMPORTS

import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
import imageio
import os
import numpy as np
from tqdm import tqdm
from IPython.display import Image, display
import contextily as ctx
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports ready")



In [ ]:
# CELL 3: CONFIG

PLACE_NAME = "Kathmandu, Nepal"

START_YEAR = 1950
END_YEAR = 2025

OUTPUT_DIR = 'kathmandu_frames'
GIF_FILE = 'kathmandu_timelapse.gif'
MP4_FILE = 'kathmandu_timelapse.mp4'
FPS = 10

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)



In [ ]:
# CELL 4: DOWNLOAD DATA

print("Downloading buildings...")

buildings = ox.features_from_place(
    PLACE_NAME,
    tags={'building': True}
)

buildings = buildings.reset_index()
buildings = buildings[['geometry']]
buildings = buildings[buildings.geometry.is_valid]
buildings = buildings[~buildings.geometry.is_empty]

print(f"✓ Buildings: {len(buildings)}")



In [ ]:
# CELL 5: PROJECT + DISTANCE
# Convert to UTM (meters)

buildings = buildings.to_crs(epsg=32645)

# Kathmandu center

CENTER_LON = 85.3072
CENTER_LAT = 27.7045

center_point = gpd.GeoSeries(
    [gpd.points_from_xy([CENTER_LON], [CENTER_LAT])[0]],
    crs="EPSG:4326"
).to_crs(epsg=32645)[0]

# Distance in meters

buildings['dist'] = buildings.geometry.centroid.distance(center_point)



In [ ]:
# CELL 6: ASSIGN YEARS (IMPROVED MODEL)

dist_norm = buildings['dist'] / buildings['dist'].max()

buildings['year'] = (
    START_YEAR +
    (dist_norm ** 1.5) * (END_YEAR - START_YEAR)
)

# Add noise

np.random.seed(42)

buildings['year'] += np.random.normal(
    0, 3, len(buildings)
)

# Earthquake effect (2015)

mask = buildings['year'] > 2015
buildings.loc[mask, 'year'] += np.random.randint(-3, 5, size=mask.sum())

buildings['year'] = buildings['year'].clip(START_YEAR, END_YEAR).astype(int)

print("✓ Years assigned")



In [ ]:
# CELL 7: MAP BOUNDS

bounds = buildings.total_bounds
x_min, y_min, x_max, y_max = bounds

pad_x = (x_max - x_min) * 0.05
pad_y = (y_max - y_min) * 0.05

x_min -= pad_x
x_max += pad_x
y_min -= pad_y
y_max += pad_y



In [ ]:
# CELL 8: GENERATE FRAMES

print("Generating frames...")

BG_COLOR = '#0d1b2a'

years = list(range(START_YEAR, END_YEAR + 1, 2)) # step=2 for speed

for year in tqdm(years):
    existing = buildings[buildings['year'] <= year]

    fig, ax = plt.subplots(figsize=(10, 10), facecolor=BG_COLOR)
    ax.set_facecolor(BG_COLOR)

    if len(existing) > 0:
        age = year - existing['year']
        existing.plot(ax=ax, column=age, cmap='plasma', linewidth=0)

    # Basemap
    try:
        ctx.add_basemap(ax, crs=buildings.crs, source=ctx.providers.CartoDB.DarkMatter)
    except:
        pass

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.axis('off')

    # Year label
    ax.text(
        0.95, 0.05, str(year),
        transform=ax.transAxes,
        color='white',
        fontsize=20,
        ha='right',
        va='bottom',
        bbox=dict(facecolor='black', alpha=0.6)
    )

    # Count
    ax.text(
        0.05, 0.05, f"{len(existing):,} buildings",
        transform=ax.transAxes,
        color='white',
        fontsize=10,
        ha='left',
        va='bottom',
        bbox=dict(facecolor='black', alpha=0.6)
    )

    # WATERMARK
    ax.text(
        0.99, 0.01,
        "@charanb",
        transform=ax.transAxes,
        color='white',
        fontsize=8,
        ha='right',
        va='bottom',
        bbox=dict(facecolor='black', alpha=0.4, pad=2)
    )

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/{year}.png', dpi=100, facecolor=BG_COLOR)
    plt.close()

print("✓ Frames done")



In [ ]:
# CELL 9: CREATE GIF + MP4
# Create GIF
imageio.mimsave(
    GIF_FILE,
    frames,
    fps=FPS
)

# Create MP4
imageio.mimsave(
    MP4_FILE,
    frames,
    fps=FPS,
    codec='libx264'
)

print("GIF and MP4 successfully generated.")

In [ ]:
# CELL 10: DISPLAY + DOWNLOAD

display(Image(filename=GIF_FILE))

from google.colab import files
files.download(GIF_FILE)
files.download(MP4_FILE)